In [ ]:
# Mission 3 TF-IDF + Logistic Regression baseline
# 공식 규정: 모든 class의 Decision Threshold는 0.5로 고정합니다.
# 모델 입력은 utterances[].text 전사 텍스트만 사용하며 startAt/id/recordId는 입력·정렬·분할에 사용하지 않습니다.
# 내부 검증은 Training 80:20 multilabel stratified split, seed=42를 사용합니다.
# 최종 CSV는 팀 공용 루트 inference.py로 생성합니다 (min-1 후처리 포함).


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (current, *current.parents) if (p / "src/dcc_data").is_dir() and (p / "scripts").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f"저장소를 준비한 뒤 그 안에서 실행하세요: {current}")
PYTHON_EXECUTABLE = os.environ.get("DCC_PYTHON", sys.executable)
if not Path(PYTHON_EXECUTABLE).is_file():
    raise FileNotFoundError(f"Python executable not found: {PYTHON_EXECUTABLE}")
try:
    from google.colab import drive as colab_drive
except ImportError:
    colab_drive = None
IN_COLAB = colab_drive is not None
if IN_COLAB and "DCC_DATA_ROOT" not in os.environ:
    colab_drive.mount("/content/drive")
if str(REPO_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "scripts"))


In [ ]:
default_data = Path("/content/drive/MyDrive/대학부 데이터") if IN_COLAB else REPO_ROOT / "data/raw_original"
DATA_DIR = Path(os.environ.get("DCC_DATA_ROOT", default_data)).expanduser().resolve()
TRAIN_LABEL_DIR = DATA_DIR / "Training/2.라벨링데이터/TL_서울_구급"
VALID_LABEL_DIR = DATA_DIR / "Validation/2.라벨링데이터/VL_서울_구급"
VALID_AUDIO_DIR = DATA_DIR / "Validation/1.원천데이터/VS_서울_구급"
for folder in (TRAIN_LABEL_DIR, VALID_LABEL_DIR, VALID_AUDIO_DIR):
    if not folder.is_dir():
        raise FileNotFoundError(f"공식 데이터 폴더를 찾지 못했습니다: {folder}")
ARTIFACT_DIR = REPO_ROOT / "artifacts"
PREDICTION_PATH = REPO_ROOT / "outputs/mission3.csv"
print("Repository:", REPO_ROOT)
print("Data:", DATA_DIR)


In [ ]:
train_json_files = sorted(TRAIN_LABEL_DIR.rglob("*.json"))
valid_json_files = sorted(VALID_LABEL_DIR.rglob("*.json"))
print("Training JSON 개수:", len(train_json_files))
print("Validation JSON 개수:", len(valid_json_files))
assert train_json_files, f"Training JSON을 찾지 못했습니다: {TRAIN_LABEL_DIR}"
assert valid_json_files, f"Validation JSON을 찾지 못했습니다: {VALID_LABEL_DIR}"


In [ ]:
import json

with train_json_files[0].open("r", encoding="utf-8-sig") as file:
    sample_data = json.load(file)

utterances = sample_data.get("utterances", [])
utterances = utterances if isinstance(utterances, list) else []
first_text = next((item.get("text", "") for item in utterances if isinstance(item, dict)), "")
print("symptom:", sample_data.get("symptom"))
print("발화 개수:", len(utterances))
print("첫 번째 발화 text:", first_text)


In [ ]:
from mission3_baseline import load_training_dataframe, make_dataset_summary

dataframe = load_training_dataframe(TRAIN_LABEL_DIR)
summary = make_dataset_summary(dataframe)
print("불러온 데이터 개수:", len(dataframe))
print("빈 전사 수:", summary["empty_text_count"])
print("목표 9개 증상이 하나도 없는 샘플 수:", summary["zero_target_label_count"])
print("JSON 읽기 실패 수:", summary["load_failure_count"])
assert summary["load_failure_count"] == 0, "읽기 실패한 Training JSON을 먼저 확인하세요."
display(dataframe.head())


In [ ]:
# 평가 대상 증상과 평가 대상에서 제외된 증상을 구분해 표시한다.
display(dataframe[["label_file_name", "symptoms", "non_target_symptoms"]].head(10))


In [ ]:
# Colab 기본 환경에 multilabel stratified split 패키지가 없을 때만 설치합니다.
import importlib.util
if importlib.util.find_spec('iterstrat') is None:
    subprocess.run([PYTHON_EXECUTABLE, '-m', 'pip', 'install', '-q', 'iterative-stratification'], check=True)


In [ ]:
# 공식 규정상 threshold는 0.5로 고정합니다.
subprocess.run([
    PYTHON_EXECUTABLE, str(REPO_ROOT / "scripts/mission3_baseline.py"),
    "--label_dir", str(TRAIN_LABEL_DIR), "--output_dir", str(ARTIFACT_DIR),
    "--threshold", "0.5", "--test_size", "0.2", "--seed", "42",
], cwd=REPO_ROOT, check=True)


In [ ]:
# 공식 Validation 정답은 사용하지 않습니다. 팀 최종 제출용 루트 inference.py로 추론합니다.
# 최종 제출 경로는 0.5 threshold + min-1 후처리이므로 별도 scripts/inference.py와 출력이 다를 수 있습니다.
subprocess.run([
    PYTHON_EXECUTABLE, str(REPO_ROOT / "inference.py"),
    "--audio_dir", str(VALID_AUDIO_DIR), "--label_dir", str(VALID_LABEL_DIR),
    "--ckpt_path", str(ARTIFACT_DIR / "mission3_tfidf_logreg.joblib"),
    "--output", str(PREDICTION_PATH),
], cwd=REPO_ROOT, check=True)


In [ ]:
with (ARTIFACT_DIR / "metrics.json").open("r", encoding="utf-8") as file:
    metrics = json.load(file)

print("Validation Macro F1:", metrics["macro_f1"])
print("Decision Threshold:", metrics["threshold"])
assert metrics["threshold"] == 0.5, "공식 규정상 threshold는 반드시 0.5여야 합니다."


In [ ]:
import pandas as pd

submission = pd.read_csv(PREDICTION_PATH)
validation_json_count = len(valid_json_files)
print("예측 행 수:", len(submission))
print("Validation JSON 수:", validation_json_count)
display(submission.head())

assert list(submission.columns) == ["label file name", "symptom"], "Mission 3 CSV 열 이름을 확인하세요."
assert len(submission) == validation_json_count, "입력 JSON 수와 출력 CSV 행 수가 다릅니다."


## 제출 전 주의

자동 검사는 코드 구조와 기본 테스트를 확인하는 것이며 실제 대회 데이터 실행을 대신하지 않습니다. 최종 제출용 학습 노트북은 공식 Training 데이터로 실제 실행한 학습 로그와 주요 결과가 남아 있는 상태로 저장하고, `inference.py` 단일 실행으로 결과 CSV가 생성되는지 다시 확인합니다.

이 노트북에서 생성하는 checkpoint는 `scripts/mission3_baseline.py` 기반입니다. 팀의 기록된 P0 재평가 점수와 직접 비교할 때는 학습 데이터 범위·스크립트·seed가 동일한지 먼저 확인하세요. 최종 CSV 추론은 저장소 루트 `inference.py` (threshold 0.5 및 팀의 min-1 규칙)를 따릅니다.
